# Payment Fraud Detection - Snowflake ML Platform Demo

## *Snowflake ML Platform Demo*

### **Use Case**: Payment Transaction Fraud Detection
Detecting fraudulent payment transactions using Snowflake's complete ML platform - Feature Store, Model Registry, and Cortex AI.

### **Business Value**:
- **Risk Reduction**: Detect fraudulent transactions before settlement
- **Customer Experience**: Minimize false positives that block legitimate spend
- **Platform**: Centralized ML infrastructure for consistency across fraud models

### **What This Demo Shows**:
- ✅ **Feature Store**: Centralized feature management eliminating duplicate engineering
- ✅ **Model Registry**: Version-controlled model lifecycle management  
- ✅ **Cortex Integration**: AI-powered analysis complementing ML
- ✅ **Real-time Scoring**: Deploy models for immediate business impact

### **Prerequisites**:
1. Environment setup completed (`01_env_setup.sql`)
2. Sample data loaded (`02_generate_and_load_data.ipynb`)
3. Running in **Snowflake Notebooks**
4. **Packages installed**: Use package selector to add:
   - `snowflake-ml-python`
   - `matplotlib` (for visualizations)

*This addresses common finserv challenges: feature consistency across fraud models, model versioning, and operationalizing ML at scale.*

## 1. Setup and Data Exploration

- Set our environment 
- Import required Snowflake ML packages.

In [ ]:
# Setup for Snowflake ML Platform

# Import required libraries (installed via package selector)
import snowflake.snowpark as snowpark
from snowflake.ml.feature_store import FeatureStore, CreationMode, Entity, FeatureView
from snowflake.ml.modeling.ensemble import RandomForestClassifier
from snowflake.ml.registry import Registry
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Get Snowflake session (works in both Snowflake Notebooks and locally)
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("Running in Snowflake Notebooks")
except:
    from snowflake_connection import get_local_session
    session = get_local_session()
    print("Running locally")

print("Connected to Snowflake ML Platform")
print(f"Account: {session.get_current_account()}")
print(f"User: {session.get_current_user()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

# Switch to demo database and schema
session.use_database("FINSERV_FRAUD_DEMO")
session.use_schema("RAW_DATA")

print(f"Working with {session.get_current_database()}.{session.get_current_schema()}")
print("="*60)

## 1.1 Sample Data
**Our demo data contains 3 tables with ~10,000 rows to mirror a real finserv payment environment**

The data includes:
- **Transactions table**: Payment transaction data with amount, merchant, device, channel, and fraud labels
- **Customers table**: Customer account profiles with tier and spending baseline
- **Merchants table**: Merchant data including category and risk tier

We'll use these to build our payment fraud detection features.

In [ ]:
%%sql -r dataframe_1
SELECT 'TRANSACTIONS' as table_name, COUNT(*) as row_count FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS
UNION ALL
SELECT 'CUSTOMERS' as table_name, COUNT(*) as row_count FROM FINSERV_FRAUD_DEMO.RAW_DATA.CUSTOMERS
UNION ALL
SELECT 'MERCHANTS' as table_name, COUNT(*) as row_count FROM FINSERV_FRAUD_DEMO.RAW_DATA.MERCHANTS

## 1.2 Understand the patterns

Looking at the transaction data to understand the distribution of payment activity. This query shows transaction volume, average amounts, and fraud rates by merchant category.

This is crucial for fraud detection because we need to understand what "normal" looks like before we can identify what's suspicious. You'll see e-commerce and travel have higher fraud rates - exactly what we'd expect in the real world.

The key insight: fraudulent transactions tend to cluster around specific merchant categories, unusual amounts relative to the customer's baseline, and cross-border activity.

In [ ]:
%%sql -r dataframe_2
SELECT 
    merchant_category,
    COUNT(*) as transaction_count,
    ROUND(AVG(amount), 2) as avg_transaction_amount,
    ROUND(MAX(amount), 2) as max_transaction_amount,
    SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) as fraud_count,
    ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 1) as fraud_rate_pct
FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS
GROUP BY merchant_category
ORDER BY transaction_count DESC

## 1.3 Feature Store
Create the Feature Store using Snowflake's ML SDK. This gives us a centralized place to store and manage all our features.

As we build this together, you'll see how it automatically connects to your existing Snowflake environment - no complex setup required.

In [ ]:
# Create Feature Store using Snowflake ML SDK 
fs = FeatureStore(
    session=session,
    database="FINSERV_FRAUD_DEMO",
    name="FINSERV_FEATURE_STORE",
    default_warehouse="STANDARD_WH_01_XS",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

print("Feature Store Schema created successfully!")

# List all schemas and filter for feature stores
all_schemas = session.sql("""
    SELECT 
        CATALOG_NAME as DATABASE,
        SCHEMA_NAME as SCHEMA,
        SCHEMA_OWNER as OWNER,
        CREATED as CREATED_DATE
    FROM INFORMATION_SCHEMA.SCHEMATA 
    WHERE CATALOG_NAME = 'FINSERV_FRAUD_DEMO'
    AND SCHEMA_NAME LIKE '%FINSERV_FEATURE%'
    ORDER BY CREATED DESC
""").collect()

for row in all_schemas:
    print(f"  - {row['DATABASE']}.{row['SCHEMA']} (Owner: {row['OWNER']})")

## 1.4 Feature Table
To set up the foundation for our feature store, we'll create an entity for customers - the core object our features will be associated with.

We're defining:
- *Entity name*: "customer" - the main entity type
- *Join keys*: "customer_id" - how we'll link features to specific customers

Next, we create the feature table structure. We're organizing features into logical groups:
- *Transaction volume features*: Count and sum of transactions in past 30 days
- *Amount features*: Average and max transaction amounts
- *Fraud indicators*: Labels for supervised training

This gives us a solid foundation to build our fraud detection features on.

In [ ]:
# Create Entity for customers
customer_entity = Entity(
    name="customer",
    join_keys=["customer_id"]
)

# Register the entity
fs.register_entity(customer_entity)
print("Customer entity registered!")


# List all entities to verify
print("\nListing all entities...")
try:
    entities_df = fs.list_entities()
    
    if entities_df.count() > 0:
        print("Found entities:")
        entities_df.show()
    else:
        print("No entities found in the feature store")
        
except Exception as e:
    print(f"Error listing entities: {str(e)}")


# Create Feature Table
# Note: is_fraudulent is NUMBER(1,0) instead of BOOLEAN for Model Monitor compatibility
session.sql("""
CREATE OR REPLACE TABLE FINSERV_FRAUD_DEMO.FINSERV_FEATURE_STORE.customer_features (
    customer_id NUMBER(38,0),
    feature_timestamp TIMESTAMP_NTZ,
    txn_count_30d NUMBER(38,0),
    avg_txn_amount_30d FLOAT,
    max_txn_ratio_30d FLOAT,
    is_fraudulent NUMBER(1,0)
)
""").collect()

print("Feature table created!")

In [ ]:
# Session cleanup and configuration for ML training
print("🔧 Preparing session for ML training...")

try:
    # First, clean up any problematic temporary stages
    print("🧹 Cleaning up temporary stages...")
    temp_stages = session.sql("SHOW STAGES LIKE 'SNOWPARK_TEMP_STAGE_%'").collect()
    
    for stage in temp_stages:
        try:
            stage_name = stage['name']
            session.sql(f"DROP STAGE IF EXISTS {stage_name}").collect()
            print(f"   Dropped temporary stage: {stage_name}")
        except:
            pass  # Ignore errors when dropping stages
    
    # Configure session for ML operations
    session.sql("USE SCHEMA FINSERV_MODEL_REGISTRY").collect()
    
    # Verify our persistent stage exists
    stage_info = session.sql("DESC STAGE ML_STAGE").collect()
    print("✅ Persistent ML stage verified: ML_STAGE")
    
    # Reset session state to clear any cached stage references
    session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()
    session.sql("ALTER SESSION SET QUERY_TAG = 'SNOWPARK_ML_FRESH_SESSION'").collect()
    
    print("✅ Session cleaned and configured for ML training")
    print(f"   Current schema: {session.get_current_schema()}")
    
except Exception as e:
    print(f"⚠️ Session configuration warning: {e}")
    print("   Proceeding anyway - if training fails, restart the notebook kernel")

print("🔄 Ready for ML model training...")

## 1.5 Features
Calculating the features from our raw transaction data, transforming payments into meaningful signals for fraud detection.

We're calculating:
- **Transaction count** in the last 30 days
- **Average transaction amount** as a spending baseline
- **Max transaction ratio** - the largest single transaction relative to the customer's average (a strong fraud signal)
- **Fraud label** based on the `is_fraud` flag we injected during data generation

Notice how we're using Snowpark, running Python inside Snowflake right where the data is.

There are additional features you could add for production:
- Velocity: transactions in the last 1h, 6h
- Cross-border ratio: % of transactions outside home country
- New merchant flag: first time transacting with this merchant
- Device change flag: different device from customer's normal pattern

This demonstrates how you can build a comprehensive feature engineering pipeline directly in Snowflake, keeping data and compute in one place.

In [ ]:
# Calculate features from transaction data
print("Calculating features...")

final_features = session.sql("""
    SELECT 
        t.customer_id,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ as feature_timestamp,
        COUNT(*) as txn_count_30d,
        AVG(t.amount) as avg_txn_amount_30d,
        CASE 
            WHEN AVG(t.amount) > 0 THEN MAX(t.amount) / AVG(t.amount)
            ELSE 0
        END as max_txn_ratio_30d,
        MAX(CASE WHEN t.is_fraud THEN 1 ELSE 0 END) as is_fraudulent
    FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS t
    WHERE t.transaction_datetime >= DATEADD(day, -30, CURRENT_DATE())
    GROUP BY t.customer_id
    ORDER BY txn_count_30d DESC
""")

# Write to feature table
final_features.write.mode("overwrite").save_as_table("FINSERV_FRAUD_DEMO.FINSERV_FEATURE_STORE.customer_features")

print("Features calculated and stored!")
print(f"Total customer feature rows: {final_features.count()}")

# Show sample
final_features.limit(5).show()

## 1.6 Feature View

Feature Views wrap our calculated features with metadata - versioning, refresh schedule, entity association, and descriptions.

**The business value for finserv teams:**
- *Centralized feature management* - no more duplicate feature engineering across fraud, AML, and credit models
- *Feature lineage and versioning* - see exactly how features were created and when they changed
- *Self-service analytics* - data scientists can discover and reuse features without going back to raw data
- *Consistency* - everyone uses the same feature definitions, eliminating discrepancies

This addresses one of the most common pain points in finserv ML: having 10+ fraud models each calculating "average transaction amount" slightly differently.

In [ ]:
# Create and register FeatureView
feature_view = FeatureView(
    name="fraud_detection_features_view",
    entities=[customer_entity],
    feature_df=session.table("FINSERV_FRAUD_DEMO.FINSERV_FEATURE_STORE.customer_features"),
    timestamp_col="feature_timestamp",
    refresh_freq="1 day",
    desc="Features for payment fraud detection"
)

# Register with versioning
registered_fv = fs.register_feature_view(
    feature_view=feature_view,
    version="1.0",
    block=True,
    overwrite=True
)

print("FeatureView registered with versioning!")
all_fvs_df = fs.list_feature_views().select('name', 'version', 'desc', 'refresh_freq')
all_fvs_df.show()

## 1.7 Model Registry

Model Registry gives us a centralized place to store, version, and manage all our ML models (both trained in Snowflake or outside).

In [ ]:
# Create Model Registry 
print("Creating Model Registry...")

# Create schema
session.sql("CREATE SCHEMA IF NOT EXISTS FINSERV_FRAUD_DEMO.FINSERV_MODEL_REGISTRY").collect()
print("Model Registry schema created!")

# Create Model Registry instance
mr = Registry(
    session=session,
    database_name="FINSERV_FRAUD_DEMO",
    schema_name="FINSERV_MODEL_REGISTRY"
)

print("Model Registry created!")

## 2. Training
Training our payment fraud detection model using Snowpark. We're using a Random Forest classifier - the entire pipeline happens in Snowflake, so data never leaves your environment.

In [ ]:
# Clean up any existing models
try:
    session.sql("DROP MODEL IF EXISTS FINSERV_FRAUD_DEMO.FINSERV_MODEL_REGISTRY.FRAUD_DETECTION_RF").collect()
    print("Dropped existing FRAUD_DETECTION_RF model")
except:
    pass

# Get training data from feature table
training_data = session.table("FINSERV_FRAUD_DEMO.FINSERV_FEATURE_STORE.customer_features")
print(f"Loaded training data: {training_data.count()} rows")

# Train Random Forest model
print("Training Random Forest model...")

# Define feature columns and label
feature_cols = ["TXN_COUNT_30D", "AVG_TXN_AMOUNT_30D", "MAX_TXN_RATIO_30D"]
label_col = "IS_FRAUDULENT"

print(f"Feature columns: {feature_cols}")
print(f"Label column: {label_col}")

import warnings
warnings.filterwarnings("ignore", message="The sample input has.*rows.*Using the first.*rows")

# Train V1 model with intentionally suboptimal hyperparameters (baseline)
print("\nTraining V1 with basic hyperparameters (baseline model)...")
rf_model = RandomForestClassifier(
    input_cols=feature_cols,
    label_cols=label_col,
    output_cols="FRAUD_PREDICTION",
    n_estimators=10,      # Very few trees (default is 100)
    max_depth=3,          # Shallow trees (default is None)
    min_samples_split=20  # Less splitting (default is 2)
)

rf_model.fit(training_data)
print("V1 Model trained (intentionally simple for baseline)!")
print("   Parameters: 10 trees, max_depth=3, min_samples_split=20")

# Make predictions
predictions = rf_model.predict(training_data)
print("Predictions generated!")
predictions.select("CUSTOMER_ID", "IS_FRAUDULENT", "FRAUD_PREDICTION").limit(5).show()

## 3. Model Versioning

Model versioning is crucial for managing your 15+ models in production. We'll register our model with version control, then create an improved version to show how easy it is to iterate and track changes.

Let's register our first model version. This gives us a baseline to compare against and shows how we can track model performance over time.

In [ ]:
# Register model with versioning
print("Registering model...")

registered_model = mr.log_model(
    model=rf_model,
    model_name="fraud_detection_rf",
    version_name="v1",
    comment="Baseline Random Forest for payment fraud detection",
    options={'relax_version': False}
)

print("✅ Model registered with versioning!")
print(f"✅ Model: {registered_model.model_name} v{registered_model.version_name}")

## 3.1 Model V2

Now let's create an improved version of our model. This simulates what happens in real life when you want to improve model performance - maybe you get more data, or you want to try different parameters. With Snowflake's Model Registry, this is incredibly easy to manage.

In [ ]:
# Create model version 2.0
print("🔄 Creating model version 2.0...")

# Train improved model with optimized hyperparameters
print("\n🔧 Training V2 with optimized hyperparameters (improved model)...")
rf_model_v2 = RandomForestClassifier(
    input_cols=["TXN_COUNT_30D", "AVG_TXN_AMOUNT_30D", "MAX_TXN_RATIO_30D"],
    label_cols="IS_FRAUDULENT",
    output_cols="FRAUD_PREDICTION",
    n_estimators=100,     # Good number of trees (10x more than V1)
    max_depth=None,       # No depth limit (vs V1's max_depth=3)
    min_samples_split=2,  # Default, allows more splitting (vs V1's 20)
    min_samples_leaf=1    # Allow fine-grained splits
)

rf_model_v2.fit(training_data)
print("✅ V2 Model trained (optimized for better performance)!")
print("   Parameters: 100 trees, no depth limit, min_samples_split=2")


# Register version 2.0
registered_model_v2 = mr.log_model(
    model=rf_model_v2,
    model_name="fraud_detection_rf",
    version_name="v2",
    comment="Improved Random Forest v2",
    options={'relax_version': False}
)

print("✅ Model version 2.0 registered!")
print(f"✅ Version: {registered_model_v2.version_name}")



# List all versions
print("\n📦 Model versions:")
model = mr.get_model("fraud_detection_rf")
versions = model.versions() 

for version in versions:
    print(f"  - {model.name} {version.version_name}")

## 4. Real-time Inference

This is where we see the complete end-to-end value. We'll take new data that our model hasn't seen before and get fraud predictions in real-time. This shows how you can deploy models and get immediate business value.

In [ ]:
# Real-time inference
print("Running real-time inference...")

# Get new data for inference (from the other 60 days, not the training period)
new_data = session.sql("""
    SELECT 
        t.customer_id,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ as feature_timestamp,
        COUNT(*) as txn_count_30d,
        AVG(t.amount) as avg_txn_amount_30d,
        CASE 
            WHEN AVG(t.amount) > 0 THEN MAX(t.amount) / AVG(t.amount)
            ELSE 0
        END as max_txn_ratio_30d,
        MAX(CASE WHEN t.is_fraud THEN 1 ELSE 0 END) as is_fraudulent
    FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS t
    WHERE t.transaction_datetime < DATEADD(day, -30, CURRENT_DATE())
    AND t.transaction_datetime >= DATEADD(day, -90, CURRENT_DATE())
    GROUP BY t.customer_id
    ORDER BY RANDOM()
    LIMIT 1000
""")

# Make predictions using our model
inference_results = rf_model_v2.predict(new_data)

# Show results
print("Inference complete!")
print(f"Processed {inference_results.count()} customers for fraud scoring")
print("Predictions on new data (from 60-90 days ago):")
inference_results.select(
    "CUSTOMER_ID", 
    "IS_FRAUDULENT", 
    "FRAUD_PREDICTION"
).limit(10).show()

In [ ]:
# Evaluation Metrics: Compare Random Forest vs Simple Rule-based Approach
print("\n📊 MODEL PERFORMANCE EVALUATION")
print("=" * 50)

# Convert to pandas for easier analysis
results_df = inference_results.to_pandas()

# Calculate confusion matrix components
true_positives = len(results_df[(results_df['IS_FRAUDULENT'] == True) & (results_df['FRAUD_PREDICTION'] == True)])
false_positives = len(results_df[(results_df['IS_FRAUDULENT'] == False) & (results_df['FRAUD_PREDICTION'] == True)])
false_negatives = len(results_df[(results_df['IS_FRAUDULENT'] == True) & (results_df['FRAUD_PREDICTION'] == False)])
true_negatives = len(results_df[(results_df['IS_FRAUDULENT'] == False) & (results_df['FRAUD_PREDICTION'] == False)])

# Calculate metrics
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (true_positives + true_negatives) / len(results_df)

# Create visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 14))

# 1. Performance Metrics Bar Chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy, precision, recall, f1_score]
colors = ['#2E8B57', '#4169E1', '#FF6347', '#FFD700']

bars1 = ax1.bar(metrics, values, color=colors, alpha=0.8)
ax1.set_title('Random Forest Model Performance', fontsize=14, fontweight='bold', pad=20)
ax1.set_ylabel('Score')
ax1.set_ylim(0, 1.1)  # Increased to give more space for value labels
ax1.grid(True, alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars1, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{value:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 2. Confusion Matrix Heatmap
confusion_data = [[true_negatives, false_positives], [false_negatives, true_positives]]
labels = ['Predicted\nLegitimate', 'Predicted\nFraudulent']
actual_labels = ['Actual\nLegitimate', 'Actual\nFraudulent']

im = ax2.imshow(confusion_data, cmap='Blues', alpha=0.8)
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
ax2.set_xticklabels(labels)
ax2.set_yticklabels(actual_labels)
ax2.set_title('Confusion Matrix', fontsize=14, fontweight='bold', pad=20)

# Add text annotations
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(confusion_data[i][j]), ha='center', va='center', 
                fontsize=16, fontweight='bold', color='white' if confusion_data[i][j] > max(max(row) for row in confusion_data)/2 else 'black')

# 3. Business Impact Pie Chart
impact_labels = ['Correctly Identified\nFraud', 'Missed Fraud\n(False Negatives)', 'False Positives\n(Legitimate Flagged)']
impact_sizes = [true_positives, false_negatives, false_positives]
impact_colors = ['#32CD32', '#FF6B6B', '#FFA500']

# Only show labels for non-zero segments to avoid overlap
filtered_labels = []
filtered_sizes = []
filtered_colors = []
for i, size in enumerate(impact_sizes):
    if size > 0:
        filtered_labels.append(impact_labels[i])
        filtered_sizes.append(size)
        filtered_colors.append(impact_colors[i])

wedges, texts, autotexts = ax3.pie(filtered_sizes, labels=filtered_labels, colors=filtered_colors, 
                                  autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
ax3.set_title('Business Impact Distribution', fontsize=14, fontweight='bold', pad=20)

# 4. Model Comparison
rf_accuracy = (results_df['IS_FRAUDULENT'] == results_df['FRAUD_PREDICTION']).mean()
simple_accuracy = 1.0  # Ground truth baseline

models = ['Ground Truth\n(Baseline)', 'Random Forest\nModel']
accuracies = [simple_accuracy, rf_accuracy]
model_colors = ['#808080', '#4CAF50']

bars4 = ax4.bar(models, accuracies, color=model_colors, alpha=0.8)
ax4.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold', pad=20)
ax4.set_ylabel('Accuracy')
ax4.set_ylim(0, 1.1)
ax4.grid(True, alpha=0.3)

# Add value labels and improvement percentage
for bar, acc in zip(bars4, accuracies):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

improvement = ((rf_accuracy - simple_accuracy) / simple_accuracy * 100)
ax4.text(0.5, 0.95, f'Improvement: {improvement:+.1f}%', 
         transform=ax4.transAxes, ha='center', fontsize=12, 
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

plt.tight_layout(pad=3.0)
plt.show()

# Print summary
print(f"\nBusiness Impact:")
print(f"   Correctly identified {true_positives} fraudulent transactions (True Positives)")
print(f"   Missed {false_negatives} fraudulent transactions (False Negatives = fraud loss exposure)")
print(f"   Flagged {false_positives} legitimate transactions incorrectly (False Positives = blocked customer spend)")

print(f"\nKey Insights:")
if rf_accuracy > simple_accuracy:
    print(f"   Random Forest outperforms simple rule-based approach")
elif rf_accuracy == simple_accuracy:
    print(f"   Random Forest and simple rule-based approach perform equally")
else:
    print(f"   Simple rule-based approach performs better on this sample")
    
print(f"   Model shows {'high' if f1_score > 0.7 else 'moderate' if f1_score > 0.5 else 'low'} performance for fraud detection")
print(f"\nNote: In finserv, precision = fewer false blocks (better customer experience)")
print(f"      Recall = fewer missed fraudulent transactions (lower fraud loss)")

## 5. Snowflake Cortex Integration

Now let's show how Cortex AI complements the ML model. Where the Random Forest gives you a binary fraud/not-fraud prediction, Cortex can add explainability - telling the fraud analyst *why* a transaction looks suspicious.

Integrating Snowflake Cortex with our ML pipeline. This gives us AI-powered analysis that can complement our traditional ML models. We'll use Cortex to analyze fraud patterns and also demonstrate churn prediction capabilities.

Let's use Cortex to analyze some athletes for fraud risk from data in the raw table. This shows how you can combine traditional ML with AI-powered analysis to get deeper insights into your data.

In [ ]:
# Snowflake Cortex LLM Analysis - Payment Fraud Risk Assessment
cortex_fraud_sql = """
SELECT 
    t.customer_id,
    c.account_tier,
    COUNT(*) as recent_txn_count,
    ROUND(AVG(t.amount), 2) as avg_amount,
    ROUND(MAX(t.amount), 2) as max_amount,
    SUM(CASE WHEN t.country_code != 'US' THEN 1 ELSE 0 END) as international_txns,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        'Analyze this customer for payment fraud risk. ' ||
        'Account tier: ' || c.account_tier || ', ' ||
        'Transactions last 30 days: ' || COUNT(*)::VARCHAR || ', ' ||
        'Average transaction: $' || ROUND(AVG(t.amount), 2)::VARCHAR || ', ' ||
        'Largest transaction: $' || ROUND(MAX(t.amount), 2)::VARCHAR || ', ' ||
        'International transactions: ' || SUM(CASE WHEN t.country_code != 'US' THEN 1 ELSE 0 END)::VARCHAR || '. ' ||
        'Rate fraud risk as LOW/MEDIUM/HIGH with a brief one-sentence explanation.'
    ) as fraud_risk_assessment
FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS t
JOIN FINSERV_FRAUD_DEMO.RAW_DATA.CUSTOMERS c ON t.customer_id = c.customer_id
WHERE t.transaction_datetime >= DATEADD(day, -30, CURRENT_DATE())
GROUP BY t.customer_id, c.account_tier
HAVING COUNT(*) >= 5
ORDER BY MAX(t.amount) / NULLIF(AVG(t.amount), 0) DESC
LIMIT 5
"""

print("Running Cortex fraud risk assessment...")
session.sql(cortex_fraud_sql).show()

In [ ]:
# Cortex vs RandomForest Model Comparison
print("Comparing Cortex LLM vs RandomForest Model")
print("=" * 60)

# Get 200 random customer IDs for comparison
random_customers = session.sql("""
    SELECT DISTINCT customer_id
    FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS 
    WHERE transaction_datetime >= DATEADD(day, -30, CURRENT_DATE())
    ORDER BY RANDOM()
    LIMIT 200
""").collect()

customer_ids = [row['CUSTOMER_ID'] for row in random_customers]
customer_ids_str = ','.join(map(str, customer_ids))

# Get Cortex fraud assessment for these customers
cortex_results = session.sql(f"""
    SELECT 
        t.customer_id,
        COUNT(*) as txn_count,
        ROUND(AVG(t.amount), 2) as avg_amount,
        ROUND(MAX(t.amount), 2) as max_amount,
        SUM(CASE WHEN t.country_code != 'US' THEN 1 ELSE 0 END) as intl_txns,
        SNOWFLAKE.CORTEX.COMPLETE(
            'llama3.1-70b',
            'Based on these payment metrics, respond TRUE if this looks fraudulent, otherwise FALSE. ' ||
            'Transactions: ' || COUNT(*)::VARCHAR || ', ' ||
            'Avg amount: $' || ROUND(AVG(t.amount), 2)::VARCHAR || ', ' ||
            'Max amount: $' || ROUND(MAX(t.amount), 2)::VARCHAR || ', ' ||
            'International transactions: ' || SUM(CASE WHEN t.country_code != 'US' THEN 1 ELSE 0 END)::VARCHAR || '. ' ||
            'Answer with TRUE or FALSE only.'
        ) as fraud_assessment
    FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS t
    WHERE t.customer_id IN ({customer_ids_str})
    AND t.transaction_datetime >= DATEADD(day, -30, CURRENT_DATE())
    GROUP BY t.customer_id
""").collect()

# Get RandomForest predictions for the same customers
rf_features = session.sql(f"""
    SELECT 
        t.customer_id,
        COUNT(*) as txn_count_30d,
        AVG(t.amount) as avg_txn_amount_30d,
        CASE 
            WHEN AVG(t.amount) > 0 THEN MAX(t.amount) / AVG(t.amount)
            ELSE 0
        END as max_txn_ratio_30d,
        MAX(CASE WHEN t.is_fraud THEN 1 ELSE 0 END) as is_fraudulent
    FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS t
    WHERE t.customer_id IN ({customer_ids_str})
    AND t.transaction_datetime >= DATEADD(day, -30, CURRENT_DATE())
    GROUP BY t.customer_id
""")

rf_predictions = rf_model_v2.predict(rf_features).collect()

# Process results
cortex_predictions_mapping = {}
for row in cortex_results:
    customer_id = row['CUSTOMER_ID']
    assessment = row['FRAUD_ASSESSMENT']
    is_fraud = 'TRUE' in assessment.upper()
    cortex_predictions_mapping[customer_id] = is_fraud

rf_predictions_mapping = {row['CUSTOMER_ID']: row['FRAUD_PREDICTION'] for row in rf_predictions}

# Create comparison DataFrame
comparison_data = []
for customer_id in customer_ids:
    cortex_prediction = cortex_predictions_mapping.get(customer_id, False)
    rf_prediction = rf_predictions_mapping.get(customer_id, False)
    comparison_data.append({
        'customer_id': customer_id,
        'cortex_fraud': cortex_prediction,
        'rf_fraud': rf_prediction
    })

df = pd.DataFrame(comparison_data)

cortex_positive = df[df['cortex_fraud'] == True].shape[0]
rf_positive = df[df['rf_fraud'] == True].shape[0]
agreement = df[df['cortex_fraud'] == df['rf_fraud']].shape[0]
agreement_rate = agreement / len(df)

print(f"\nCOMPARISON RESULTS:")
print(f"   Customers analyzed: {len(df)}")
print(f"   Agreement rate: {agreement_rate:.3f} ({agreement}/{len(df)})")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

methods = ['Cortex LLM', 'RandomForest']
flagged_counts = [cortex_positive, rf_positive]

bars = ax1.bar(methods, flagged_counts, color=['#FF6B6B', '#4ECDC4'])
ax1.set_title('Fraud Detection Comparison\n(200 Random Customers)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Customers Flagged as Fraudulent')
ax1.set_ylim(0, max(flagged_counts) * 1.2 if max(flagged_counts) > 0 else 10)

for bar, count in zip(bars, flagged_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             str(count), ha='center', va='bottom', fontweight='bold')

agreement_labels = ['Agreement', 'Disagreement']
agreement_sizes = [agreement, len(df) - agreement]
colors = ['#90EE90', '#FFB6C1']

ax2.pie(agreement_sizes, labels=agreement_labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title(f'Method Agreement\n({agreement_rate:.1%} agreement)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nKEY INSIGHTS:")
print(f"   Methods agree on {agreement_rate:.1%} of cases")
print(f"   The ML model provides fast batch scoring; Cortex adds human-readable explanations for analyst review")

Cortex can also be used for additional risk analysis - for example, summarizing a customer's recent transaction behavior for a fraud analyst case review.

In [ ]:
# Snowflake Cortex LLM - Case Summary for Fraud Analyst Review
cortex_fraud_sql = """
SELECT 
    t.customer_id,
    c.account_tier,
    c.account_age_days,
    COUNT(*) as recent_txns,
    ROUND(SUM(t.amount), 2) as total_spend,
    SUM(CASE WHEN t.country_code != 'US' THEN 1 ELSE 0 END) as intl_txns,
    ROUND(MAX(t.amount), 2) as largest_txn,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        'Write a 2-sentence fraud case summary for an analyst. Customer details: ' ||
        'Account tier: ' || c.account_tier || ', ' ||
        'Account age: ' || c.account_age_days::VARCHAR || ' days, ' ||
        'Transactions last 30 days: ' || COUNT(*)::VARCHAR || ', ' ||
        'Total spend: $' || ROUND(SUM(t.amount), 2)::VARCHAR || ', ' ||
        'International transactions: ' || SUM(CASE WHEN t.country_code != 'US' THEN 1 ELSE 0 END)::VARCHAR || ', ' ||
        'Largest single transaction: $' || ROUND(MAX(t.amount), 2)::VARCHAR || '.'
    ) as analyst_case_summary
FROM FINSERV_FRAUD_DEMO.RAW_DATA.TRANSACTIONS t
JOIN FINSERV_FRAUD_DEMO.RAW_DATA.CUSTOMERS c ON t.customer_id = c.customer_id
WHERE t.transaction_datetime >= DATEADD(day, -30, CURRENT_DATE())
GROUP BY t.customer_id, c.account_tier, c.account_age_days
ORDER BY ROUND(MAX(t.amount), 2) / NULLIF(ROUND(AVG(t.amount), 2), 0) DESC
LIMIT 5
"""

print("Running Cortex analyst case summaries...")
session.sql(cortex_fraud_sql).show()

## 7. Summary

**What we demonstrated in this demo:**

**Feature Store**: Centralized feature management eliminates duplicate engineering across fraud, AML, and credit risk models. All features - transaction velocity, amount ratios, cross-border signals - live in one place with versioning and lineage.

**Model Registry**: Version-controlled model lifecycle. V1 (baseline) vs V2 (optimized) comparison shows how easy it is to iterate and track improvements without losing audit history.

**Model Versioning**: Side-by-side version comparison makes it straightforward to validate improvements before promoting to production.

**Real-time Inference**: End-to-end scoring pipeline that can be operationalized immediately - no separate inference infrastructure needed.

**Cortex Integration**: LLM-powered explanations complement the ML model. The Random Forest gives you fast batch scoring; Cortex adds human-readable reasoning for analyst review queues.

**The key value for finserv teams**: Instead of each fraud, AML, and credit team maintaining separate feature pipelines with inconsistent definitions, you have one unified ML platform. One source of truth for features. Full model audit trail for regulatory compliance. All inside your existing Snowflake environment.

In [ ]:
'''
# Clean up existing schemas for fresh demo re-run

try:
    fs = FeatureStore(
        session=session,
        database="FINSERV_FRAUD_DEMO",
        name="FINSERV_FEATURE_STORE",
        default_warehouse="STANDARD_WH_01_XS",
        creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
    )
    fs.delete_entity("customer")
    print("Customer entity deleted!")
except Exception as e:
    print(f"Customer entity not found: {str(e)}")


print("\nCleaning up existing schemas...")
try:
    session.sql("DROP SCHEMA IF EXISTS FINSERV_FRAUD_DEMO.FINSERV_FEATURE_STORE CASCADE").collect()
    print("Dropped existing FINSERV_FEATURE_STORE")
except:
    pass

try:
    session.sql("DROP SCHEMA IF EXISTS FINSERV_FRAUD_DEMO.FINSERV_MODEL_REGISTRY CASCADE").collect()
    print("Dropped existing FINSERV_MODEL_REGISTRY")
except:
    pass

try:
    session.sql("DROP MODEL IF EXISTS FINSERV_FRAUD_DEMO.FINSERV_MODEL_REGISTRY.FRAUD_DETECTION_RF").collect()
    print("Dropped existing FRAUD_DETECTION_RF model")
except:
    pass

'''

In [ ]:
'''
model = mr.get_model("fraud_detection_rf")
# Delete specific versions
model.delete_version("v2") 
'''